In [24]:
dataset_version="25M_demo"

Cargamos los conjuntos de entrenamiento y validación

In [18]:
from datasets import load_from_disk, DatasetDict
import os


BASE_PATH=f"data/{dataset_version}"
train_ds = load_from_disk(os.path.join(BASE_PATH, "train"))
validation_ds=load_from_disk(os.path.join(BASE_PATH, "validation"))

Creamos el formato de conversación con el que entrena MedGemma. Creamos una nueva columna con la conversacion con la que sera entrenado el modelo para cada par imagen-texto

In [12]:
def formatting_message(example):
    example["message"] = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image", #la imagen que se le pasa
                },
                {
                    "type": "text",
                    "text": PROMPT,
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text":example["caption"],
                },
            ],
        },
    ]
    return example
   
    

In [ ]:
#PROMPT="Describe this brain MRI scan in detail "
#train_ds_small= train_ds.select(range(10))
#formatted_train_small = train_ds_small.map(formatting_message)
# formatted_train_small["message"]

In [ ]:

PROMPT="Describe this brain MRI scan in detail "
formatted_train = train_ds.map(formatting_message)
formatted_validation=validation_ds.map(formatting_message)

Column(["The image is a CT scan of the brain, showing the intricate structures within the cranial cavity. The region of interest, located in the left-center horizontally and upper-middle vertically, occupies approximately 1.6% of the area. This region exhibits an abnormality, potentially indicative of a disease, characterized by a difference in texture or density compared to the surrounding brain tissue. The abnormality's proximity to critical brain structures suggests a possible relationship, where the affected area could be influencing or being influenced by the adjacent tissues, potentially impacting their function or being a result of a localized pathological process.", "This CT scan of the brain shows a region of interest located centrally and in the lower-middle area, with an area ratio of 1.3%. The region of interest is characterized by an abnormality that differs in density from the surrounding brain tissue, which could indicate the presence of a hemorrhage. The abnormality's p

Cargamos el modelo medGemma

In [34]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# Check if GPU supports bfloat16
cache_dir="C:/Users/pedro/Proyectos/Medical_VLM/picasso/modelo"

model_kwargs = dict(
    attn_implementation="eager",
    
    device_map="auto",
)

model = AutoModelForImageTextToText.from_pretrained(cache_dir,local_files_only=True, **model_kwargs)
processor = AutoProcessor.from_pretrained(cache_dir,local_files_only=True,)

# Use right padding to avoid issues during training
processor.tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Usamos Low-Rank Adaptation (LoRA), a parameter-efficient fine-tuning method

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear", #Busca automáticamente todas las capas lineales del modelo y les pone un adaptador Lora
    task_type="CAUSAL_LM", #Tipico de LLMs normales. Predice la siguiente palabra basada en las anteriores
    modules_to_save=[ #estos modulos tambien van a ser entrenados (creo que se podrían quitar)
        "lm_head", # ultima capa que predice la siguiente palabra
        "embed_tokens", # pasa el texto a tokens
    ],
)



Definimos un collator para tratar con imagenes y texto en el entrenamiento

In [ ]:
def collate_fn(examples: list[dict[str, any]]):
    texts = []
    images = []
    for example in examples:
        images.append([example["image"]])
        texts.append(
            processor.apply_chat_template(
                example["message"], add_generation_prompt=False, tokenize=False
            ).strip()
        )

    # Tokenize the texts and process the images
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # The labels are the input_ids, with the padding and image tokens masked in
    # the loss computation
    labels = batch["input_ids"].clone() #Es la entrada completa. Token de la imagen, la pregunta, padding... 

    # Mask image tokens
    # Identifica el número exacto (ID) que representa a la "Imagen" dentro del vocabulario del modelo
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    
    # Mask tokens that are not used in the loss computation
    
    #-100 significa que se ignore para calcular la perdida
    labels[labels == processor.tokenizer.pad_token_id] = -100  #Mask the padding
    labels[labels == image_token_id] = -100 #Mask the image
    labels[labels == 262144] = -100 #262144 es un numero específico de la arquitectura Gemma/PaliGemma. Mask it.

    batch["labels"] = labels
    return batch



<table style="width:100%; border-collapse: collapse; border: 1px solid #ddd;">
    <thead>
        <tr style="background-color: #f2f2f2;">
            <th style="border: 1px solid #ddd; padding: 10px; text-align: left; width: 15%;">Concepto</th>
            <th style="border: 1px solid #ddd; padding: 10px; text-align: left; width: 35%;">Contenido (Simplificado)</th>
            <th style="border: 1px solid #ddd; padding: 10px; text-align: left; width: 50%;">Función</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid #ddd; padding: 10px;"><strong><code>input_ids</code></strong></td>
            <td style="border: 1px solid #ddd; padding: 10px; font-family: monospace;"><code>[&lt;IMAGEN&gt;, Es, un, glioma]</code></td>
            <td style="border: 1px solid #ddd; padding: 10px;">Es la <strong>secuencia completa</strong> (imagen + texto) que el modelo <strong>LEE</strong> y utiliza como contexto.</td>
        </tr>
        <tr>
            <td style="border: 1px solid #ddd; padding: 10px;"><strong><code>labels</code></strong></td>
            <td style="border: 1px solid #ddd; padding: 10px; font-family: monospace;"><code>[-100, Es, un, glioma]</code></td>
            <td style="border: 1px solid #ddd; padding: 10px;">Es la <strong>respuesta correcta</strong> que el modelo <strong>DEBE GENERAR</strong>. El valor <code>-100</code> indica a PyTorch que ignore ese token y no lo evalúe.</td>
        </tr>
    </tbody>
</table>

Argumentos de entrenamiento

In [22]:
from trl import SFTConfig

args = SFTConfig(
    output_dir=f"medgemma-FT-MedTrinity{dataset_version}",            
    num_train_epochs=1,                       
    per_device_train_batch_size=8,                           
    per_device_eval_batch_size=8,                            
    gradient_accumulation_steps=8,                           
    gradient_checkpointing=True,                             
    optim="adamw_torch_fused",                               
    logging_steps=0.1,                                        
    save_strategy="epoch",                                   
    eval_strategy="steps",                                   
    eval_steps=0.1,                                           
    learning_rate=2e-4,                             
    bf16=True,                                               
    max_grad_norm=0.3,                                       
    warmup_ratio=0.03,                                       
    lr_scheduler_type="linear",                              
    push_to_hub=True,                                        
    report_to="none",
    gradient_checkpointing_kwargs={"use_reentrant": False},  
    dataset_kwargs={"skip_prepare_dataset": True},           
    remove_unused_columns = False, #obligas a mantener todas las columnas (incluida la imagen) para que lleguen al collator                      
    label_names=["labels"],                                  
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=formatted_train,
    eval_dataset=formatted_validation.shuffle().select(range(50)), 
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

## Entrenamiento del modelo

In [ ]:
trainer.train()

In [ ]:
trainer.save_model()